# 01. Exploratory Data Analysis (EDA)
## PRG4206 Individual Assignment: AI-Generated Text Detection System

This notebook presents the exploratory data analysis of the **HC3 (Human ChatGPT Comparison Corpus)** benchmark dataset.

### Objectives:
1. Verify dataset structure, schema, and sample counts
2. Programmatically compute human vs. AI class distribution
3. Analyze text length, word count, and character distributions
4. Inspect domain distributions across Q&A categories
5. Detect missing values, duplicates, and outliers


In [ ]:
import sys
from pathlib import Path
# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_hc3
from src.config import FIGURES_DIR

sns.set_theme(style="whitegrid", palette="muted")
print("Imports loaded successfully.")


### 1. Load Dataset
We load the preprocessed flat HC3 dataset or download it via `load_hc3()`.


In [ ]:
df = load_hc3()
print(f"Total samples : {len(df):,}")
print(f"Columns       : {list(df.columns)}")
df.head(5)


### 2. Class Distribution Analysis
Checking balance between human-written (label 0) and AI-generated (label 1).


In [ ]:
class_counts = df["label_name"].value_counts()
print("Class Counts:\n", class_counts)
print("\nProportions:\n", df["label_name"].value_counts(normalize=True))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(class_counts.index, class_counts.values, color=["#3498db", "#e74c3c"])
ax.bar_label(bars, fmt="%d")
ax.set_title("HC3 Class Distribution (Human vs AI)")
ax.set_ylabel("Number of Samples")
plt.tight_layout()
plt.show()


### 3. Text Length Distribution
Comparing word count distributions across classes.


In [ ]:
df.groupby("label_name")[["word_count", "char_count"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [("human", "#3498db"), ("ai", "#e74c3c")]:
    sub = df[df["label_name"] == label]["word_count"]
    axes[0].hist(sub.clip(0, 800), bins=40, alpha=0.6, label=f"{label} (median={sub.median():.0f})", color=color)

axes[0].set_title("Word Count Distribution (Clipped at 800 words)")
axes[0].set_xlabel("Words")
axes[0].legend()

sns.boxplot(data=df, x="label_name", y="word_count", palette={"human": "#3498db", "ai": "#e74c3c"}, showfliers=False, ax=axes[1])
axes[1].set_title("Word Count Boxplot (Outliers Omitted)")
plt.tight_layout()
plt.show()


### 4. Domain / Source Category Breakdown


In [ ]:
domain_dist = df.groupby(["source", "label_name"]).size().unstack(fill_value=0)
print(domain_dist)

domain_dist.plot(kind="bar", figsize=(10, 5), color=["#e74c3c", "#3498db"])
plt.title("Domain-Wise Breakdown of Human vs AI Samples")
plt.ylabel("Count")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


### 5. Key Findings
- **Sample Count**: Total clean usable records = 79,331.
- **Class Balance**: 66.9% Human (53,086) vs 33.1% AI (26,245) — a ~2:1 ratio.
- **Length Differences**: AI text exhibits tighter variance (std ~60 words) around a median of 174 words, whereas human texts show high skewness and variance (median 84 words, max >7,000 words).
- **Domain Dominance**: `reddit_eli5` constitutes ~78% of the data; domain-stratified evaluation is crucial.
